<a href="https://colab.research.google.com/github/ssk-algoverse/sae-binding/blob/main/circuit/sweep_ckpt_circuit_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformer_lens circuitsvis

In [ ]:
from IPython.display import clear_output

In [ ]:
import torch
from huggingface_hub import hf_hub_download
from transformer_lens import HookedTransformer, HookedTransformerConfig, utils
import numpy as np
import pandas as pd
import ast
from torch.utils.data import Dataset, DataLoader
import torch
from functools import partial
import circuitsvis as cv
from IPython.display import display, Markdown
import matplotlib.pyplot as plt
import plotly.express as px

In [ ]:
# util plotting function
def imshow(
    tensor,
    xlabel="X",
    ylabel="Y",
    zlabel=None,
    xticks=None,
    yticks=None,
    c_midpoint=0.0,
    c_scale="RdBu",
    show=True,
    **kwargs
):
    tensor = utils.to_numpy(tensor)
    n_rows, n_cols = tensor.shape

    labels = {"x": xlabel, "y": ylabel}
    if zlabel is not None:
        labels["color"] = zlabel

    # Build the figure with numeric axes
    fig = px.imshow(
        tensor,
        labels=labels,
        color_continuous_midpoint=c_midpoint,
        color_continuous_scale=c_scale,
        **kwargs
    )

    if xticks is not None:
        xtxt = [str(x) for x in xticks]
        fig.update_xaxes(
            tickmode="array",
            tickvals=list(range(n_cols)),
            ticktext=xtxt,
            type="linear",   # ensure numeric axis, not categorical
            tickangle=-90 # Add this line to rotate x-axis labels
        )

    if yticks is not None:
        ytxt = [str(y) for y in yticks]
        fig.update_yaxes(
            tickmode="array",
            tickvals=list(range(n_rows)),
            ticktext=ytxt,
            type="linear"
        )
    fig.show()
    return fig

In [ ]:
from huggingface_hub import hf_hub_download

REPO_ID = "sebastianhoenig/4L2H_Model" # "sojup/entity_binding_test"
FILENAME = "D256_L4_H2_attnOnly1_lr5.0e-04_wd0.01.pt" # "D256_L3_H2_attnOnly1_lr5.0e-04_wd0.01.pt"

weights_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)
clear_output(wait=True)

In [ ]:
REPO_ID = "sojup/entity_binding_test"
FILENAME = "id_to_entity.csv"

id_mapping_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)
clear_output(wait=True)

In [ ]:
E = 100
T = 10
D_VOCAB = E + T + 3

In [ ]:
N_LAYERS = 4
HEADS = 2

d_model = 256
n_ctx   = 64

def build_model(n_layers: int, n_heads: int) -> HookedTransformer:
    if d_model % n_heads != 0:
        return None
    d_head = d_model // n_heads

    cfg = HookedTransformerConfig(
        n_layers=n_layers,
        n_heads=n_heads,
        d_model=d_model,
        d_head=d_head,
        n_ctx=n_ctx,
        d_vocab=D_VOCAB,
        d_vocab_out=E,
        attn_only=True,
        normalization_type="LN",
        positional_embedding_type="rotary",
    )
    return HookedTransformer(cfg)

model = build_model(N_LAYERS, HEADS)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
# Load the model
pretrained_weights = torch.load(weights_path, map_location=device, weights_only=True)
state_dict = pretrained_weights["model"]
model.load_state_dict(state_dict)

print("Model loaded successfully.")

In [ ]:
# need this for attention patching later
model.cfg.use_attn_result = True

In [ ]:
id_mapping_df = pd.read_csv(id_mapping_path)
id_to_entity = dict(zip(id_mapping_df['id'], id_mapping_df['name']))

In [ ]:
id_to_entity[100] = 'loves'
id_to_entity[101] = 'works with'
id_to_entity[102] = 'interacts with'
id_to_entity[103] = 'lives with'
id_to_entity[104] = 'has a grudge against'
id_to_entity[105] = 'is interested in'
id_to_entity[106] = 'plays with'
id_to_entity[107] = 'goes to school with'
id_to_entity[108] = 'is jealous of'
id_to_entity[109] = 'wants'

In [ ]:
# class EntityBindingDataset(Dataset):
#     def __init__(self, dataframe, parse_tokens_if_str=True):
#         self.df = dataframe.reset_index(drop=True)
#         self.parse_tokens_if_str = parse_tokens_if_str

#     def __len__(self):
#         return len(self.df)

#     def __getitem__(self, idx):
#         row = self.df.iloc[idx]
#         seq = row["tokens"]
#         tokens = torch.tensor(seq, dtype=torch.long)
#         label  = torch.tensor(int(row["label"]), dtype=torch.long)
#         return tokens, label

In [ ]:
# from datasets import load_dataset

# dataset = load_dataset("sojup/entity_binding", split="test")
# clear_output()


import numpy as np
import pandas as pd
from tqdm import tqdm

E = 100  # num entities
T = 10   # num types/relations

SEP = E + T
Q = E + T + 1
PAD = E + T + 2
D_VOCAB = E + T + 3

IGNORE_INDEX = -100
ENTITIES = np.arange(0, E)
TYPES    = np.arange(E, E + T)

N_WORLDS = 80_000
MIN_FACTS, MAX_FACTS = 4, 8
SEED = 0

rng = np.random.default_rng(SEED)

def produce_example_by_index(idx: int, *, allow_self_loops: bool = False):
    rng = np.random.default_rng(np.random.SeedSequence([BASE_SEED, idx]))

    k = int(rng.integers(MIN_FACTS, MAX_FACTS + 1))

    facts = []
    seen_head_rel = set()
    while len(facts) < k:
        e = int(rng.integers(0, E))
        t = int(rng.integers(0, T)) + E
        if (e, t) in seen_head_rel:
            continue
        e2 = int(rng.integers(0, E))
        while (not allow_self_loops) and e2 == e:
            e2 = int(rng.integers(0, E))
        seen_head_rel.add((e, t))
        facts.append((e, t, e2))

    q_idx = int(rng.integers(0, k))
    Eq, Tq, E2q = facts[q_idx]

    if rng.random() < 0.75 and len(facts) < MAX_FACTS:
        distractor_t = int(rng.integers(0, T)) + E

        while distractor_t == Tq: # Ensure the relation is different
            distractor_t = int(rng.integers(0, T)) + E

        distractor_e2 = int(rng.integers(0, E))
        while distractor_e2 == E2q: # Ensure the tail is different
            distractor_e2 = int(rng.integers(0, E))

        # Add the distractor fact IF it doesn't create a collision
        if (Eq, distractor_t) not in seen_head_rel:
            distractor_fact = (Eq, distractor_t, distractor_e2)

            insert_pos = int(rng.integers(0, len(facts) + 1))
            facts.insert(insert_pos, distractor_fact)

    seq = []
    for (e, t, e2) in facts:
        seq.extend([e, t, e2, SEP])

    #if random.random() < 0.5
    seq.extend([Tq, Eq, Q])
    #else:
    #    seq.extend([Eq, Tq, Q])

    label = E2q
    return seq, label

In [ ]:
TOTAL_TRAIN = 16_100_000
BLOCK_SIZE  = 80_000
VAL_SIZE = 20_000
TRAIN_OFFSET = VAL_SIZE
TRAIN_SIZE   = TOTAL_TRAIN
BASE_SEED = 0

class ValDataset(torch.utils.data.Dataset):
    def __len__(self): return VAL_SIZE
    def __getitem__(self, i):
        seq, label = produce_example_by_index(i)
        return torch.tensor(seq, dtype=torch.long), torch.tensor(label, dtype=torch.long)


class TrainStream(torch.utils.data.IterableDataset):
    def __init__(self, block_size=BLOCK_SIZE, offset=TRAIN_OFFSET, size=TRAIN_SIZE):
        super().__init__()
        self.block_size = block_size
        self.offset = offset
        self.size = size
        self._epoch = 0

    def set_epoch(self, epoch:int):
        self._epoch = epoch

    def __iter__(self):
        # compute which block to serve this epoch, with wrap-around
        start_in_train = (self._epoch * self.block_size) % self.size
        # stream exactly block_size samples each epoch
        for i in range(self.block_size):
            local_idx = (start_in_train + i) % self.size
            global_idx = self.offset + local_idx
            seq, label = produce_example_by_index(global_idx)
            x = torch.tensor(seq, dtype=torch.long)
            y = torch.tensor(label, dtype=torch.long)
            yield x, y

    def __len__(self):
        return self.block_size

val_dataset = ValDataset()
train_dataset = TrainStream()


In [ ]:
# from datasets import Dataset, DatasetDict

# # Create Dataset objects from pandas DataFrames
# train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
# val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))
# test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

# # Create a DatasetDict
# dataset_dict = DatasetDict({
#     'train': train_dataset,
#     'validation': val_dataset,
#     'test': test_dataset
# })

# print(dataset_dict)
  # from huggingface_hub import notebook_login
  # notebook_login()

  # repo_name = "sojup/entity_binding"

  # # You will need to be logged in to Hugging Face to push to the hub


  # # Push the dataset to the Hugging Face Hub
  # dataset_dict.push_to_hub(repo_name)

  # print(f"DatasetDict created. Uncomment the notebook_login() and push_to_hub() lines and replace '{repo_name}' with your desired repository name to upload to the Hugging Face Hub.")

In [ ]:
# test_df = dataset.to_pandas()
# test_dataset = EntityBindingDataset(test_df)

In [ ]:
example, label = val_dataset[0]
example, label, [id_to_entity[i.item()] for i in example], id_to_entity[label.item()]

In [ ]:
## considering the incorrect token to be the 10th most probable prediction
## take a batch of examples with this heuristic

In [ ]:
### Attention Pattern

In [ ]:
logits, cache = model.run_with_cache(example)

In [ ]:
probs = torch.softmax(logits[0, -1, :], dim=-1)

In [ ]:
for i, idx in enumerate(torch.topk(probs, 5).indices.tolist()):
  print(f"{id_to_entity[idx]} {torch.topk(probs, 5).values.tolist()[i]}")

print(f"\n\nCorrect label: {id_to_entity[label.item()]}")

In [ ]:
### Taking Michelle as corrupt ex.

In [ ]:
### Method 1: Residual stream patching

In [ ]:
example

In [ ]:
id_to_entity_rev = {v: k for k, v in id_to_entity.items()}

In [ ]:
corrupt_answer_token, clean_answer_token = id_to_entity_rev["Jeremy"], id_to_entity_rev["Christopher"]


# corrupt_answer_token, clean_answer_token = id_to_entity_rev[top_2_predictions_list[1]["name"]], id_to_entity_rev[top_2_predictions_list[0]["name"]]
# corrupt_example = example.clone()
# corrupt_example[corrupt_example == clean_answer_token] = corrupt_answer_token

In [ ]:
corrupt_answer_token, clean_answer_token

In [ ]:
corrupt_example = example.clone()
corrupt_example[corrupt_example == clean_answer_token] = corrupt_answer_token

In [ ]:
example

In [ ]:
corrupt_example

In [ ]:
corrupt_logits, corrupt_cache = model.run_with_cache(corrupt_example)

In [ ]:
corrupt_probs = torch.softmax(corrupt_logits[0, -1, :], dim=-1)

In [ ]:
for i, idx in enumerate(torch.topk(corrupt_probs, 5).indices.tolist()):
  print(f"{id_to_entity[idx]} {torch.topk(corrupt_probs, 5).values.tolist()[i]}")

print(f"\n\nCorrect label: {id_to_entity[label.item()]}")

### Activation Patching

At each hook point - so before and after each transformer block, we patch in the activations of the clean example - and observe the logit difference between the two targets

We have two different things we can patch in to see how information travels in the network.
Since our triplets are <E1, R, E2>, we can produce a corrupt example that has a different (corrupt) E2 - and then patch the activations of the clean cache back in - or ask for another relation by corrupting Tq, Eq

In [ ]:
cache["blocks.0.hook_resid_pre"].shape, corrupt_cache["blocks.0.hook_resid_pre"].shape

In [ ]:
layers = ["blocks.0.hook_resid_pre", *[f"blocks.{i}.hook_resid_post" for i in range(model.cfg.n_layers)]]
n_layers = len(layers)
n_pos = len(example)
clean_answer_token = 88
corrupt_answer_token = 66

def patch_residual_stream(activations, hook, layer, pos):
   activations[:, pos, :] = cache[layer][:, pos, :]
   return activations

In [ ]:
clean_score   = logits[0, -1, clean_answer_token] - logits[0, -1, corrupt_answer_token]
corrupt_score = corrupt_logits[0, -1, clean_answer_token] - corrupt_logits[0, -1, corrupt_answer_token]

In [ ]:
clean_score, corrupt_score

In [ ]:
patching_effect = torch.zeros(n_layers, n_pos)

for l, layer in enumerate(layers):
    for pos in range(n_pos):
        fwd_hooks = [(layer, partial(patch_residual_stream, layer=layer, pos=pos))]
        prediction_logits = model.run_with_hooks(corrupt_example,
                                                 fwd_hooks=fwd_hooks)[0, -1]
        # Uncomment to get intuition
        #print(f"For layer {l} and position {pos} the logit of Christine is {prediction_logits[clean_answer_token]}\
        #and the logit of Michelle is {prediction_logits[corrupt_answer_token]}")
        patch_score = prediction_logits[clean_answer_token] - prediction_logits[corrupt_answer_token]
        # if this is 1, then we have fully recovered the clean_score with our patching. If its 0, then we are at the corrupt score
        patching_effect[l, pos] = (patch_score - corrupt_score) / (clean_score - corrupt_score + 1e-9)

In [ ]:
patching_effect

In [ ]:
imshow(patching_effect, xticks=[id_to_entity[i.item()] for i in example], yticks=layers, xlabel="pos", ylabel="layer",
       zlabel="Percentage clean logit recovered", title="Patching corrupt Entity with clean Entity", width=1000, height=380)

Now lets do the same with our second example. Patching out "is interested in"

In [ ]:
corrupt_relation_example = example.clone()
corrupt_relation_example[corrupt_relation_example == id_to_entity_rev["is interested in"]] = id_to_entity_rev["plays with"]

In [ ]:
corrupt_relation_logits, corrupt_relation_cache = model.run_with_cache(corrupt_relation_example)

In [ ]:
corrupt_relation_probs = torch.softmax(corrupt_relation_logits[0, -1, :], dim=-1)

In [ ]:
for i, idx in enumerate(torch.topk(corrupt_relation_probs, 5).indices.tolist()):
  print(f"{id_to_entity[idx]} {torch.topk(corrupt_relation_probs, 5).values.tolist()[i]}")

print(f"\n\nCorrect label: {id_to_entity[label.item()]}")

-> Just from the log probs we can already guess that the relation is not being attended to. We can later investigate a second example, where the entity appears twice - and then see the effect of the relation there.

I will actually come back to this, because here its not easy to find a "counter-entity example" - and we would not see anything either way

Now lets patch out "is interested in Jason", for "loves Lindsay"

In [ ]:
corrupt_example2 = example.clone()
corrupt_example2[17] = id_to_entity_rev["Lindsay"]
corrupt_example2[16] = id_to_entity_rev["loves"]
corrupt_logits2, corrupt_cache2 = model.run_with_cache(corrupt_example2)
corrupt_probs2 = torch.softmax(corrupt_logits2[0, -1, :], dim=-1)
for i, idx in enumerate(torch.topk(corrupt_probs2, 5).indices.tolist()):
  print(f"{id_to_entity[idx]} {torch.topk(corrupt_probs2, 5).values.tolist()[i]}")

print(f"\n\nCorrect label: {id_to_entity[label.item()]}")

In [ ]:
[id_to_entity[i.item()] for i in corrupt_example2]

In [ ]:
clean_answer_token = 37
corrupt_answer_token = id_to_entity_rev["Susan"]

In [ ]:
clean_score   = logits[0, -1, clean_answer_token] - logits[0, -1, corrupt_answer_token]
corrupt_score2 = corrupt_logits2[0, -1, clean_answer_token] - corrupt_logits2[0, -1, corrupt_answer_token]
clean_score, corrupt_score2

In [ ]:
logits[0, -1, corrupt_answer_token]

In [ ]:
patching_effect = torch.zeros(n_layers, n_pos)

for l, layer in enumerate(layers):
    for pos in range(n_pos):
        fwd_hooks = [(layer, partial(patch_residual_stream, layer=layer, pos=pos))]
        prediction_logits = model.run_with_hooks(corrupt_example2,
                                                 fwd_hooks=fwd_hooks)[0, -1]
        # Uncomment to get intuition
        #print(f"For layer {l} and position {pos} the logit of Christine is {prediction_logits[clean_answer_token]}\
        #and the logit of Susan is {prediction_logits[corrupt_answer_token]}")
        patch_score = prediction_logits[clean_answer_token] - prediction_logits[corrupt_answer_token]
        # if this is 1, then we have fully recovered the clean_score with our patching. If its 0, then we are at the corrupt score
        patching_effect[l, pos] = (patch_score - corrupt_score2) / (clean_score - corrupt_score2)

In [ ]:
imshow(patching_effect, xticks=[id_to_entity[i.item()] for i in example], yticks=layers, xlabel="pos", ylabel="layer",
       zlabel="Percentage clean logit recovered", title="Patching corrupt Tq,Eq with clean Tq,Eq", width=1000, height=380)

-> It looks like the "Question-Entity" information travels in the block1 to the last token

### This was the layer-aggregated view. We can now dive deeper into how this work is split up between the individual heads

We simply patch differently. Instead of patching the before/after each transformer block as defined above (layers = ["blocks.0.hook_resid_pre", *[f"blocks.{i}.hook_resid_post" for i in range(model.cfg.n_layers)]]), we now patch after the result of individual heads gets added to the residual stream


![image.png](attachment:92fb0e1a-90f9-4a48-8872-48465a4c4ddd.png)![image.png](attachment:3ef1b165-68af-4b8f-bde4-44d09a23297b.png)

In [ ]:
def patch_head_result(activations, hook, layer=None, head=None, pos=None):
   activations[:, pos, head, :] = cache[hook.name][:, pos, head, :]
   return activations

#### Lets start with the first example again - where we patched out Christine for Michelle

In [ ]:
clean_answer_token = 37
corrupt_answer_token = 99

clean_score   = logits[0, -1, clean_answer_token] - logits[0, -1, corrupt_answer_token]
corrupt_score = corrupt_logits[0, -1, clean_answer_token] - corrupt_logits[0, -1, corrupt_answer_token]
clean_score, corrupt_score

In [ ]:
n_layers = model.cfg.n_layers
n_heads = model.cfg.n_heads
n_pos = len(example)


patching_effect = torch.zeros(n_layers*n_heads, n_pos)
for layer in range(n_layers):
    for head in range(n_heads):
        for pos in range(n_pos):
            fwd_hooks = [(
            	f"blocks.{layer}.attn.hook_result",
	            partial(patch_head_result, layer=layer, head=head, pos=pos)
            )]
            prediction_logits = model.run_with_hooks(corrupt_example,
                                                     fwd_hooks=fwd_hooks)[0, -1]
            #print(f"For layer {layer}, head {head} in pos {pos}, the prediction logit for Chrstine is {prediction_logits[clean_answer_token]}\
            #and for Michelle its {prediction_logits[corrupt_answer_token]}")
            patch_score = prediction_logits[clean_answer_token] - prediction_logits[corrupt_answer_token]
            patching_effect[n_heads*layer+head, pos] = (patch_score - corrupt_score) / (clean_score - corrupt_score)


token_labels = [f"(pos {i:2}) {t}" for i, t in enumerate(example)]
layerhead_labels = [f"{l}.{h}" for l in range(n_layers) for h in range(n_heads)]
imshow(patching_effect, xticks=[id_to_entity[i.item()] for i in example], yticks=layerhead_labels, xlabel="position", ylabel="layer.head",
           zlabel="Logit difference", title=f"Patching with Michelle instead of Christine", width=1000, height=800)

#### We can now also do this, when we corrupt via the Tq, Eq

In [ ]:
clean_answer_token = 37
corrupt_answer_token = id_to_entity_rev["Susan"]
clean_score   = logits[0, -1, clean_answer_token] - logits[0, -1, corrupt_answer_token]
corrupt_score2 = corrupt_logits2[0, -1, clean_answer_token] - corrupt_logits2[0, -1, corrupt_answer_token]
clean_score, corrupt_score2

In [ ]:
n_layers = model.cfg.n_layers
n_heads = model.cfg.n_heads
n_pos = len(example)


patching_effect = torch.zeros(n_layers*n_heads, n_pos)
for layer in range(n_layers):
    for head in range(n_heads):
        for pos in range(n_pos):
            fwd_hooks = [(
            	f"blocks.{layer}.attn.hook_result",
	            partial(patch_head_result, layer=layer, head=head, pos=pos)
            )]
            prediction_logits = model.run_with_hooks(corrupt_example2, fwd_hooks=fwd_hooks)[0, -1]
            patch_score = prediction_logits[clean_answer_token] - prediction_logits[corrupt_answer_token]
            patching_effect[n_heads*layer+head, pos] = (patch_score - corrupt_score2) / (clean_score - corrupt_score2)

token_labels = [f"(pos {i:2}) {t}" for i, t in enumerate(example)]
layerhead_labels = [f"{l}.{h}" for l in range(n_layers) for h in range(n_heads)]
imshow(patching_effect, xticks=[id_to_entity[i.item()] for i in example], yticks=layerhead_labels, xlabel="position", ylabel="layer.head",
           zlabel="Logit difference", title=f"Patching with loves Linday instead of is interested in Jason", width=1000, height=800)

-> We now see that it is specifically head 1.1, which appears to do the heavy lifting here. Attending to the "relation" token

### Attention Pattern

In [ ]:
from IPython.display import display, Markdown
import numpy as np, torch, circuitsvis as cv

def tensor_to_numpy(t):
    if isinstance(t, torch.Tensor):
        t = t.detach().cpu().numpy()
    return t

def attn_for_cv(t):
    t = tensor_to_numpy(t)
    if t.ndim == 4:  # [batch, heads, seq, seq]
        if t.shape[0] != 1:
            raise ValueError(f"Batch dim {t.shape[0]} != 1; pass a single example.")
        t = t[0]
    if t.ndim != 3:
        raise ValueError(f"Expected 3D [heads, seq, seq], got {t.shape}")
    return t

str_tokens = [id_to_entity[i.item()] for i in example]
for layer in range(model.cfg.n_layers):
    raw = cache["pattern", layer]
    attn = attn_for_cv(raw)
    if len(str_tokens) != attn.shape[-1]:
        raise ValueError(f"Token length {len(str_tokens)} != seq_len {attn.shape[-1]}")
    display(Markdown(f"### Layer {layer}"))
    display(cv.attention.attention_patterns(tokens=str_tokens, attention=attn))


### Failure cases

In [ ]:
# target_prefixes = [torch.tensor([37, 108, 65, 110], dtype=torch.long),
#                  torch.tensor([4, 109, 35, 110], dtype=torch.long),
#                  torch.tensor([44, 100, 91, 110], dtype=torch.long),
#                  torch.tensor([75, 101, 72, 110], dtype=torch.long)]
# train_dataset = load_dataset("sojup/entity_binding", split="train")
# train_df = train_dataset.to_pandas()
# train_dataset = EntityBindingDataset(train_df)
# failure_cases = []
# for tokens, label in train_dataset:
#   for target_prefix in target_prefixes:
#     if len(tokens) >= len(target_prefix) and torch.equal(tokens[:len(target_prefix)], target_prefix):
#         failure_cases.append((tokens, label))

In [ ]:
for example, label in train_dataset:
  logits, cache = model.run_with_cache(example)
  probs = torch.softmax(logits[0, -1, :], dim=-1)
  pred = torch.argmax(probs).item()
  if pred == label.item():
    continue
  print(example, label.item(), pred)

  for i, idx in enumerate(torch.topk(probs, 5).indices.tolist()):
    print(f"{id_to_entity[idx]} {torch.topk(probs, 5).values.tolist()[i]}")

  # Store top 2 predictions in a list
  top_2_values, top_2_indices = torch.topk(probs, 2)
  top_2_predictions_list = []
  for i in range(2):
      idx = top_2_indices[i].item()
      name = id_to_entity[idx]
      prob = top_2_values[i].item()
      top_2_predictions_list.append({"index": idx, "name": name, "probability": prob})
  print(f"\n\nCorrect label: {id_to_entity[label.item()]}")
  id_to_entity_rev = {v: k for k, v in id_to_entity.items()}
  corrupt_answer_token, clean_answer_token = id_to_entity_rev[top_2_predictions_list[1]["name"]], id_to_entity_rev[top_2_predictions_list[0]["name"]]
  # id_to_entity_rev["Michelle"], id_to_entity_rev["Christine"]
  corrupt_example = example.clone()
  corrupt_example[corrupt_example == clean_answer_token] = corrupt_answer_token
  # corrupt_example[corrupt_example == id_to_entity_rev["Christine"]] = id_to_entity_rev["Michelle"]
  corrupt_logits, corrupt_cache = model.run_with_cache(corrupt_example)
  corrupt_probs = torch.softmax(corrupt_logits[0, -1, :], dim=-1)
  for i, idx in enumerate(torch.topk(corrupt_probs, 5).indices.tolist()):
    print(f"{id_to_entity[idx]} {torch.topk(corrupt_probs, 5).values.tolist()[i]}")

  print(f"\n\nCorrect label: {id_to_entity[label.item()]}")

  layers = ["blocks.0.hook_resid_pre", *[f"blocks.{i}.hook_resid_post" for i in range(model.cfg.n_layers)]]
  n_layers = len(layers)
  n_pos = len(example)
  # clean_answer_token = 37
  # corrupt_answer_token = 99

  def patch_residual_stream(activations, hook, layer, pos):
    activations[:, pos, :] = cache[layer][:, pos, :]
    return activations


  clean_score   = logits[0, -1, clean_answer_token] - logits[0, -1, corrupt_answer_token]
  corrupt_score = corrupt_logits[0, -1, clean_answer_token] - corrupt_logits[0, -1, corrupt_answer_token]
  print(clean_score, corrupt_score)
  patching_effect = torch.zeros(n_layers, n_pos)

  for l, layer in enumerate(layers):
      for pos in range(n_pos):
          fwd_hooks = [(layer, partial(patch_residual_stream, layer=layer, pos=pos))]
          prediction_logits = model.run_with_hooks(corrupt_example,
                                                  fwd_hooks=fwd_hooks)[0, -1]
          # Uncomment to get intuition
          patch_score = prediction_logits[clean_answer_token] - prediction_logits[corrupt_answer_token]
          # if this is 1, then we have fully recovered the clean_score with our patching. If its 0, then we are at the corrupt score
          patching_effect[l, pos] = (patch_score - corrupt_score) / (clean_score - corrupt_score)
  imshow(patching_effect, xticks=[id_to_entity[i.item()] for i in example], yticks=layers, xlabel="pos", ylabel="layer",
        zlabel="Percentage clean logit recovered", title="Patching corrupt Entity with clean Entity", width=1000, height=380)

  corrupt_relation_example = example.clone()
  corrupt_relation_example[corrupt_relation_example == id_to_entity_rev["is interested in"]] = id_to_entity_rev["plays with"]
  corrupt_relation_logits, corrupt_relation_cache = model.run_with_cache(corrupt_relation_example)
  corrupt_relation_probs = torch.softmax(corrupt_relation_logits[0, -1, :], dim=-1)
  for i, idx in enumerate(torch.topk(corrupt_relation_probs, 5).indices.tolist()):
    print(f"{id_to_entity[idx]} {torch.topk(corrupt_relation_probs, 5).values.tolist()[i]}")

  print(f"\n\nCorrect label: {id_to_entity[label.item()]}")

  clean_score   = logits[0, -1, clean_answer_token] - logits[0, -1, corrupt_answer_token]
  corrupt_score = corrupt_logits[0, -1, clean_answer_token] - corrupt_logits[0, -1, corrupt_answer_token]
  clean_score, corrupt_score

  n_layers = model.cfg.n_layers
  n_heads = model.cfg.n_heads
  n_pos = len(example)

  def patch_head_result(activations, hook, layer=None, head=None, pos=None):
    activations[:, pos, head, :] = cache[hook.name][:, pos, head, :]
    return activations
  patching_effect = torch.zeros(n_layers*n_heads, n_pos)
  for layer in range(n_layers):
      for head in range(n_heads):
          for pos in range(n_pos):
              fwd_hooks = [(
                f"blocks.{layer}.attn.hook_result",
                partial(patch_head_result, layer=layer, head=head, pos=pos)
              )]
              prediction_logits = model.run_with_hooks(corrupt_example,
                                                      fwd_hooks=fwd_hooks)[0, -1]
              patch_score = prediction_logits[clean_answer_token] - prediction_logits[corrupt_answer_token]
              patching_effect[n_heads*layer+head, pos] = (patch_score - corrupt_score) / (clean_score - corrupt_score)


  token_labels = [f"(pos {i:2}) {t}" for i, t in enumerate(example)]
  layerhead_labels = [f"{l}.{h}" for l in range(n_layers) for h in range(n_heads)]
  imshow(patching_effect, xticks=[id_to_entity[i.item()] for i in example], yticks=layerhead_labels, xlabel="position", ylabel="layer.head",
            zlabel="Logit difference", title=f"Patching with {top_2_predictions_list[1]['name']} instead of {top_2_predictions_list[0]['name']}", width=1000, height=800)

  from IPython.display import display, Markdown
  import numpy as np, torch, circuitsvis as cv

  def tensor_to_numpy(t):
      if isinstance(t, torch.Tensor):
          t = t.detach().cpu().numpy()
      return t

  def attn_for_cv(t):
      t = tensor_to_numpy(t)
      if t.ndim == 4:  # [batch, heads, seq, seq]
          if t.shape[0] != 1:
              raise ValueError(f"Batch dim {t.shape[0]} != 1; pass a single example.")
          t = t[0]
      if t.ndim != 3:
          raise ValueError(f"Expected 3D [heads, seq, seq], got {t.shape}")
      return t

  str_tokens = [id_to_entity[i.item()] for i in example]
  for layer in range(model.cfg.n_layers):
      raw = cache["pattern", layer]
      attn = attn_for_cv(raw)
      if len(str_tokens) != attn.shape[-1]:
          raise ValueError(f"Token length {len(str_tokens)} != seq_len {attn.shape[-1]}")
      display(Markdown(f"### Layer {layer}"))
      display(cv.attention.attention_patterns(tokens=str_tokens, attention=attn))
  print("*" * 50)


### Other Test data examples

In [ ]:
for i, (example, label) in enumerate(val_dataset):
  if i < 2:
    continue
  if i == 6:
    break
  logits, cache = model.run_with_cache(example)
  probs = torch.softmax(logits[0, -1, :], dim=-1)


  for i, idx in enumerate(torch.topk(probs, 5).indices.tolist()):
    print(f"{id_to_entity[idx]} {torch.topk(probs, 5).values.tolist()[i]}")

  # Store top 2 predictions in a list
  top_2_values, top_2_indices = torch.topk(probs, 2)
  top_2_predictions_list = []
  for i in range(2):
      idx = top_2_indices[i].item()
      name = id_to_entity[idx]
      prob = top_2_values[i].item()
      top_2_predictions_list.append({"index": idx, "name": name, "probability": prob})
  print(f"\n\nCorrect label: {id_to_entity[label.item()]}")
  id_to_entity_rev = {v: k for k, v in id_to_entity.items()}
  corrupt_answer_token, clean_answer_token = id_to_entity_rev[top_2_predictions_list[1]["name"]], id_to_entity_rev[top_2_predictions_list[0]["name"]]
  # id_to_entity_rev["Michelle"], id_to_entity_rev["Christine"]
  corrupt_example = example.clone()
  corrupt_example[corrupt_example == clean_answer_token] = corrupt_answer_token
  # corrupt_example[corrupt_example == id_to_entity_rev["Christine"]] = id_to_entity_rev["Michelle"]
  corrupt_logits, corrupt_cache = model.run_with_cache(corrupt_example)
  corrupt_probs = torch.softmax(corrupt_logits[0, -1, :], dim=-1)
  for i, idx in enumerate(torch.topk(corrupt_probs, 5).indices.tolist()):
    print(f"{id_to_entity[idx]} {torch.topk(corrupt_probs, 5).values.tolist()[i]}")

  print(f"\n\nCorrect label: {id_to_entity[label.item()]}")

  layers = ["blocks.0.hook_resid_pre", *[f"blocks.{i}.hook_resid_post" for i in range(model.cfg.n_layers)]]
  n_layers = len(layers)
  n_pos = len(example)
  # clean_answer_token = 37
  # corrupt_answer_token = 99

  def patch_residual_stream(activations, hook, layer, pos):
    activations[:, pos, :] = cache[layer][:, pos, :]
    return activations


  clean_score   = logits[0, -1, clean_answer_token] - logits[0, -1, corrupt_answer_token]
  corrupt_score = corrupt_logits[0, -1, clean_answer_token] - corrupt_logits[0, -1, corrupt_answer_token]
  print(clean_score, corrupt_score)
  patching_effect = torch.zeros(n_layers, n_pos)

  for l, layer in enumerate(layers):
      for pos in range(n_pos):
          fwd_hooks = [(layer, partial(patch_residual_stream, layer=layer, pos=pos))]
          prediction_logits = model.run_with_hooks(corrupt_example,
                                                  fwd_hooks=fwd_hooks)[0, -1]
          # Uncomment to get intuition
          patch_score = prediction_logits[clean_answer_token] - prediction_logits[corrupt_answer_token]
          # if this is 1, then we have fully recovered the clean_score with our patching. If its 0, then we are at the corrupt score
          patching_effect[l, pos] = (patch_score - corrupt_score) / (clean_score - corrupt_score)
  imshow(patching_effect, xticks=[id_to_entity[i.item()] for i in example], yticks=layers, xlabel="pos", ylabel="layer",
        zlabel="Percentage clean logit recovered", title="Patching corrupt Entity with clean Entity", width=1000, height=380)

  corrupt_relation_example = example.clone()
  corrupt_relation_example[corrupt_relation_example == id_to_entity_rev["is interested in"]] = id_to_entity_rev["plays with"]
  corrupt_relation_logits, corrupt_relation_cache = model.run_with_cache(corrupt_relation_example)
  corrupt_relation_probs = torch.softmax(corrupt_relation_logits[0, -1, :], dim=-1)
  for i, idx in enumerate(torch.topk(corrupt_relation_probs, 5).indices.tolist()):
    print(f"{id_to_entity[idx]} {torch.topk(corrupt_relation_probs, 5).values.tolist()[i]}")

  print(f"\n\nCorrect label: {id_to_entity[label.item()]}")

  clean_score   = logits[0, -1, clean_answer_token] - logits[0, -1, corrupt_answer_token]
  corrupt_score = corrupt_logits[0, -1, clean_answer_token] - corrupt_logits[0, -1, corrupt_answer_token]
  clean_score, corrupt_score

  n_layers = model.cfg.n_layers
  n_heads = model.cfg.n_heads
  n_pos = len(example)


  patching_effect = torch.zeros(n_layers*n_heads, n_pos)
  for layer in range(n_layers):
      for head in range(n_heads):
          for pos in range(n_pos):
              fwd_hooks = [(
                f"blocks.{layer}.attn.hook_result",
                partial(patch_head_result, layer=layer, head=head, pos=pos)
              )]
              prediction_logits = model.run_with_hooks(corrupt_example,
                                                      fwd_hooks=fwd_hooks)[0, -1]
              patch_score = prediction_logits[clean_answer_token] - prediction_logits[corrupt_answer_token]
              patching_effect[n_heads*layer+head, pos] = (patch_score - corrupt_score) / (clean_score - corrupt_score)


  token_labels = [f"(pos {i:2}) {t}" for i, t in enumerate(example)]
  layerhead_labels = [f"{l}.{h}" for l in range(n_layers) for h in range(n_heads)]
  imshow(patching_effect, xticks=[id_to_entity[i.item()] for i in example], yticks=layerhead_labels, xlabel="position", ylabel="layer.head",
            zlabel="Logit difference", title=f"Patching with {top_2_predictions_list[1]['name']} instead of {top_2_predictions_list[0]['name']}", width=1000, height=800)

  from IPython.display import display, Markdown
  import numpy as np, torch, circuitsvis as cv

  def tensor_to_numpy(t):
      if isinstance(t, torch.Tensor):
          t = t.detach().cpu().numpy()
      return t

  def attn_for_cv(t):
      t = tensor_to_numpy(t)
      if t.ndim == 4:  # [batch, heads, seq, seq]
          if t.shape[0] != 1:
              raise ValueError(f"Batch dim {t.shape[0]} != 1; pass a single example.")
          t = t[0]
      if t.ndim != 3:
          raise ValueError(f"Expected 3D [heads, seq, seq], got {t.shape}")
      return t

  str_tokens = [id_to_entity[i.item()] for i in example]
  for layer in range(model.cfg.n_layers):
      raw = cache["pattern", layer]
      attn = attn_for_cv(raw)
      if len(str_tokens) != attn.shape[-1]:
          raise ValueError(f"Token length {len(str_tokens)} != seq_len {attn.shape[-1]}")
      display(Markdown(f"### Layer {layer}"))
      display(cv.attention.attention_patterns(tokens=str_tokens, attention=attn))
  print("*" * 50)
